<a href="https://colab.research.google.com/github/gopalstud86/GenAI-Assignments/blob/main/Bronze%20Badge%20Assignments/Problem5-RAG%2BLLM%20with%20Streamlit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import streamlit as st
from langchain_community.llms import Ollama
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.chains import RetrievalQA
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from huggingface_hub import login

#login(token=" ")
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

llm = Ollama(model="llama3")

_ = llm.invoke("warmup")
texts = ["Startup check"]
#**_VWEqhXViefrePmPZXDOqxEllmqvCHOFhRj


#pdf_file = "https://raw.githubusercontent.com/gopalstud86/GenAI-Assignments/refs/heads/main/Bronze%20Badge%20Assignments/Datasets/Policy.pdf"

pdf_file = ("Policy.pdf")
# Load PDF
loader = PyPDFLoader(pdf_file)
documents = loader.load()

# Split into chunks
splitter = CharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = splitter.split_documents(documents)


# Chroma vector store
db = Chroma.from_documents(docs, embeddings)
retriever = db.as_retriever()

qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff"
)

if "messages" not in st.session_state:
    st.session_state.messages = []
if "chat_input" not in st.session_state:
    st.session_state.chat_input = ""

answer = " "
def handle_submit():
    user_text = st.session_state.chat_input.strip()
    if user_text:
        response = qa.invoke(user_text)
        answer = response["result"]
        st.session_state.messages.append(("You", user_text))
        st.session_state.messages.append(("Policy Baba", answer))
        st.session_state.chat_input = ""

st.title("Policy & Claims Copilot")
st.subheader("Welcome! You are chatting with Policy Baba")
user_query = st.text_input("Enter your claim query:", key="chat_input", on_change=handle_submit)

pre_checks = []
if "hospitalization" in user_query.lower():
    pre_checks.append("Hospitalization must be ≥ 24 hours.")
if "maternity" in user_query.lower():
    pre_checks.append("Maternity covered only after 2 years waiting period, limit ₹50,000.")
if "dental" in user_query.lower():
    pre_checks.append("Dental treatment is excluded.")

if pre_checks:
    st.markdown("## Pre-check Validation")
    for rule in pre_checks:
        st.write("- " + rule)

for sender, msg in st.session_state.messages:
    if sender == "You":
        st.markdown(f"**You:** {msg}")
    else:
        st.markdown(f"**Policy Baba says:** {msg}")




ModuleNotFoundError: No module named 'streamlit'